# Colab Optimized Fine-Tuning with Auto-Resume

This notebook is specifically designed for Google Colab to handle the longest 100,000 sentences of the WMT19 dataset.

**Key Features:**
1. **Google Drive Integration**: Saves checkpoints directly to your Google Drive so you don't lose them when Colab disconnects.
2. **Hugging Face Hub Sync**: Pushes your checkpoints to the Hugging Face Hub during training.
3. **Auto-Resume**: If Colab disconnects, simply run all cells again. It will detect the latest checkpoint in your Google Drive and resume training exactly where it left off.

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub

In [ ]:
import os

output_dir = "/kaggle/working/wmt-multilingual-lora"
print(f"✅ Kaggle environment detected. Checkpoints will be saved to: {output_dir}")
os.makedirs(output_dir, exist_ok=True)


In [ ]:
from huggingface_hub import login

# 2. Login to Hugging Face
hf_token = input("Enter your Hugging Face WRITE Token: ")
login(token=hf_token)

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

# --- CONFIGURATION ---
model_id = "CohereForAI/aya-expanse-8b"
hub_model_id = "YOUR_HF_USERNAME/aya-expanse-8b-wmt-zh-en" # ⚠️ CHANGE THIS to your actual HF username!

In [ ]:
print("Loading datasets...")
from datasets import load_dataset, concatenate_datasets

def get_dataset_safe(repo, lang_pair, target_size=30000):
    try:
        ds = load_dataset(repo, lang_pair, split="train")
    except Exception as e:
        print(f"Could not load {lang_pair} from {repo}, trying fallback. Error: {e}")
        return None
        
    def calculate_length(example):
        langs = lang_pair.split('-')
        return {"length": len(example["translation"][langs[0]])}
        
    ds = ds.map(calculate_length, num_proc=8)
    ds = ds.filter(lambda x: 20 < x["length"] < 1500)
    ds = ds.shuffle(seed=42)
    
    end_idx = min(target_size, len(ds))
    return ds.select(range(0, end_idx))

print("Loading and formatting WMT19 English to Simplified Chinese (zh-en)...")
ds_zh_en = get_dataset_safe("wmt19", "zh-en")

def format_zh(batch):
    formatted = []
    for t in batch["translation"]:
        formatted.append(f"Translate from English to Simplified Chinese:\nEnglish: {t['en']}\nChinese: {t['zh']}")
    return {"text": formatted}
train_zh_en = ds_zh_en.map(format_zh, batched=True, remove_columns=ds_zh_en.column_names)
print(f"zh-en sampled size: {len(train_zh_en)}")

print("Loading and formatting OPUS-100 English to Arabic (ar-en)...")
ds_en_ar = get_dataset_safe("opus100", "ar-en")
if ds_en_ar is None:
    ds_en_ar = get_dataset_safe("opus100", "en-ar")

def format_ar(batch):
    formatted = []
    for t in batch["translation"]:
        formatted.append(f"Translate from English to Arabic:\nEnglish: {t['en']}\nArabic: {t['ar']}")
    return {"text": formatted}
train_en_ar = ds_en_ar.map(format_ar, batched=True, remove_columns=ds_en_ar.column_names)
print(f"en-ar sampled size: {len(train_en_ar)}")

print("Loading and formatting OPUS-100 Czech to German (cs-de)...")
ds_cs_de = get_dataset_safe("opus100", "cs-de")
if ds_cs_de is None:
    ds_cs_de = get_dataset_safe("opus100", "de-cs")

def format_cs(batch):
    formatted = []
    for t in batch["translation"]:
        formatted.append(f"Translate from Czech to German:\nCzech: {t['cs']}\nGerman: {t['de']}")
    return {"text": formatted}
train_cs_de = ds_cs_de.map(format_cs, batched=True, remove_columns=ds_cs_de.column_names)
print(f"cs-de sampled size: {len(train_cs_de)}")

print("Combining datasets...")
train_dataset = concatenate_datasets([train_zh_en, train_en_ar, train_cs_de]).shuffle(seed=42)
print(f"Combined Dataset ready! Size: {len(train_dataset)}")
print(f"Sample Prompt:\n{train_dataset[0]['text']}\n")


In [ ]:
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

import gc
import torch
torch.cuda.empty_cache()
gc.collect()

print("Loading Model in 4-bit precision...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)
model.config.torch_dtype = torch.float16
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

import os
PREV_LORA = "/kaggle/input/datasets/anishracherla06/arya-english-chineese"
if os.path.exists(PREV_LORA) and os.path.isfile(os.path.join(PREV_LORA, "adapter_config.json")):
    print(f"🔄 Loading PREVIOUS LoRA weights from: {PREV_LORA}")
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, PREV_LORA, is_trainable=True)
else:
    print(f"⚠️ Previous LoRA not found at {PREV_LORA}. Falling back to a FRESH LoRA adapter.")
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)


In [ ]:
from trl import SFTConfig, SFTTrainer
import torch


# Emptying CUDA cache just in case previous cells left garbage
torch.cuda.empty_cache()

# Provide ALL training, packing, and data arguments directly to SFTConfig.
sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,      # Drop to 1 (Essential for Kaggle Dual T4 limits)
    gradient_accumulation_steps=16,     # Increase to 16 to keep same batch behavior
    gradient_checkpointing=True,
    
    # 📉 MEMORY FIX: Added custom checkpointing kwargs specifically for reducing activation memory
    gradient_checkpointing_kwargs={'use_reentrant': False}, 
    
    optim="paged_adamw_8bit", # 📉 MEMORY FIX: 8-bit optimizer
    save_steps=60,             
    save_total_limit=2,         
    logging_steps=50,
    learning_rate=2e-4,
    fp16=False, # 📉 MEMORY FIX: Disabled to completely bypass GradScaler BFloat16 bug,                  
    max_steps=120,             
    warmup_steps=15,            
    group_by_length=True,       # ⚡ CHANGED: Pairs same-length sentences together for extreme speed!
    lr_scheduler_type="cosine",
    push_to_hub=False,          
    report_to="none",
    
    dataset_text_field="text",
    packing=False,              # ⚡ CHANGED: No more cutting! Pure, flawless translation pairs.
    max_length=512, # 📉 MEMORY FIX: Hard cap at 512 tokens
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset, 
    processing_class=tokenizer,
    args=sft_config,
)

# --- RESUME LOGIC ---
last_checkpoint = None
if os.path.isdir(output_dir):
    from transformers.trainer_utils import get_last_checkpoint
    last_checkpoint = get_last_checkpoint(output_dir)

print("\n" + "="*40)
if last_checkpoint is not None:
    print(f"🚀 CHECKPOINT FOUND! Resuming training from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🌟 No previous checkpoint found. Starting fresh from step 0.")
    trainer.train()

# Final Save
trainer.model.save_pretrained(output_dir)
print(f"🎉 Training complete! Model saved perfectly to: {output_dir}")

In [ ]:
# unbabel-comet is horribly incompatible with the newest transformers library (4.49+)! 
# We MUST violently downgrade transformers to an older stable version (4.44.2) for comet to import!
# Since we already finished training the model, it is perfectly safe to downgrade.
!pip install -q evaluate unbabel-comet pytorch-lightning "transformers<4.45"


In [ ]:
import torch
import tarfile
import urllib.request
import tempfile
import os
import json
from tqdm.auto import tqdm

print("Downloading FLORES-200 Evaluation Dataset directly from Meta...")
# Because 'trust_remote_code' and 'datasets' loader scripts are now largely deprecated in newer
# HF environments, we bypass the library entirely and pull the pure FLORES-200 tar.gz directly.

samples_to_eval = 200
sources = []
references = []

# Download and extract the pure text sentences in memory safely
with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz", tmp.name)
    with tarfile.open(tmp.name, "r:gz") as tar:
        eng_file = tar.extractfile("./flores200_dataset/dev/eng_Latn.dev")
        zho_file = tar.extractfile("./flores200_dataset/dev/zho_Hans.dev")
        
        # Read exactly the first 200 samples
        sources = [line.decode("utf-8").strip() for line in eng_file.readlines()][:samples_to_eval]
        references = [line.decode("utf-8").strip() for line in zho_file.readlines()][:samples_to_eval]

# Clean up
os.remove(tmp.name)

predictions = []

print(f"Generating translations for {samples_to_eval} samples using the trained LoRA model...")
model.eval()

# We already translated it on your system previously! To avoid waiting another 12 minutes,
# we will just recalculate the predictions if they were lost during an error!
for i, src_text in enumerate(tqdm(sources, desc="Translating")):
    prompt_eval = f"Translate from English to Simplified Chinese:\nEnglish: {src_text}\nChinese:"
    inputs_eval = tokenizer(prompt_eval, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs_eval = model.generate(
            **inputs_eval, 
            max_new_tokens=150, 
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False # Greedy decoding is preferred for precise translation benchmarking
        )
        
    num_gen = outputs_eval.shape[1] - inputs_eval.input_ids.shape[1]
    pred = tokenizer.decode(outputs_eval[0][-num_gen:], skip_special_tokens=True).strip().replace('\n', ' ')
    predictions.append(pred)

print("\nCreating pure Python script to bypass CLI errors...")

# 🚀 INCREDIBLE BYPASS: The CLI crashed because of an argparse python core bug on Kaggle.
# So we will write a literal pure python script to disk that has a completely isolated memory space,
# run it as a standalone app, and print out the result safely.

data = [
    {"src": src, "mt": mt, "ref": ref}
    for src, mt, ref in zip(sources, predictions, references)
]
with open("data_to_grade.json", "w", encoding="utf-8") as f:
    json.dump(data, f)

isolated_script = """
import json
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

from comet import download_model, load_from_checkpoint

print("Downloading COMET model quietly...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

with open("data_to_grade.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Scoring translations...")
comet_results = comet_model.predict(data, batch_size=8, gpus=1)

print("="*40)
print(f"🌟 Final COMET Score (200 Samples): {comet_results.system_score:.4f}")
print("="*40)
"""

with open("run_comet.py", "w", encoding="utf-8") as f:
    f.write(isolated_script)
    
!python run_comet.py

In [ ]:
import shutil
from IPython.display import FileLink
import os

folder_path = output_dir
total_size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, filenames in os.walk(folder_path) for f in filenames)
print(f"The LoRA model raw folder size is: {total_size / (1024 * 1024):.2f} MB")

zip_path = "/kaggle/working/wmt-multilingual-lora-saved"
shutil.make_archive(zip_path, 'zip', folder_path)
print("\n✅ Click the link below to download your fine-tuned model directly to your laptop!")
display(FileLink(r'wmt-multilingual-lora-saved.zip'))
